In [6]:
import tkinter as tk
from tkinter import ttk, messagebox
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

WINDOW_TITLE = "Methane Production Prediction Software 2"

INPUT_NAMES = [
    "C (%)", "H (%)", "O (%)", "N (%)", "S (%)",
    "Ash (%)", "Solid Content (%)",
    "HTT-T (°C)", "HTT-RT (min)",
    "Dilution ratio",
    "AD-T (°C)", "AD-Time (day)"
]

OUTPUT_NAMES = [
    "Methane yield (ml/g COD)",
    "Methane production rate (ml/(g COD·d))"
]


def train_model_from_excel(excel_path: str):
    # 1) 读数据
    data = pd.read_excel(excel_path)

    # 2) 强制转成数值（非数值变 NaN）
    data = data.apply(pd.to_numeric, errors="coerce")

    # 3) 检查列数
    if data.shape[1] < 14:
        raise ValueError(
            f"Excel 列数不足：当前 {data.shape[1]} 列，但需要至少 14 列（12 inputs + 2 outputs）。\n"
            f"请确认 Dilution ratio 已加入且输出两列在第 13-14 列位置。"
        )

    # 4) 取 X / Y
    X = data.iloc[:, 0:12].values
    Y = data.iloc[:, 12:14].values

    # 5) 删除任何含 NaN 的行（X 或 Y 有 NaN 都删）
    mask = np.isfinite(X).all(axis=1) & np.isfinite(Y).all(axis=1)
    X = X[mask]
    Y = Y[mask]

    if X.shape[0] < 5:
        raise ValueError("有效样本太少（清洗 NaN 后不足 5 行），请检查 Excel 数据是否有大量空值/非数值。")

    # 6) 划分训练/测试
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # 7) 用你返稿里那套最佳参数（你上面写的是这组）
    model = RandomForestRegressor(
        n_estimators=256,
        max_depth=37,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    return model


class PredictorGUI:
    def __init__(self, root, model):
        self.root = root
        self.model = model

        self.root.title(WINDOW_TITLE)
        self.root.geometry("1050x680")

        container = ttk.Frame(root, padding=12)
        container.pack(fill="both", expand=True)

        input_frame = ttk.LabelFrame(container, text="Inputs", padding=12)
        input_frame.pack(fill="both", expand=True)

        output_frame = ttk.LabelFrame(container, text="Outputs", padding=12)
        output_frame.pack(fill="x", pady=(12, 0))

        left_col = ttk.Frame(input_frame)
        right_col = ttk.Frame(input_frame)
        left_col.grid(row=0, column=0, sticky="nsew", padx=(0, 16))
        right_col.grid(row=0, column=1, sticky="nsew")

        input_frame.columnconfigure(0, weight=1)
        input_frame.columnconfigure(1, weight=1)

        # 左侧：生物质特征
        g1 = ttk.LabelFrame(left_col, text="Biomass characteristics", padding=10)
        g1.pack(fill="both", expand=True, pady=(0, 12))

        # 右侧：HTT / Dilution / AD（Dilution 单独拆出来）
        g2 = ttk.LabelFrame(right_col, text="HTT conditions", padding=10)
        g3 = ttk.LabelFrame(right_col, text="Dilution", padding=10)  # ✅ 新增：单独的 Dilution 区域
        g4 = ttk.LabelFrame(right_col, text="AD parameters", padding=10)

        g2.pack(fill="x", pady=(0, 12))
        g3.pack(fill="x", pady=(0, 12))
        g4.pack(fill="x")

        self.entries = [None] * 12

        def add_row(parent, idx, label_text, r):
            ttk.Label(parent, text=label_text).grid(row=r, column=0, sticky="w", pady=6, padx=(0, 10))
            ent = ttk.Entry(parent, width=26)
            ent.grid(row=r, column=1, sticky="ew", pady=6)
            parent.columnconfigure(1, weight=1)
            self.entries[idx] = ent

        # 组1：0-6
        for r, i in enumerate(range(0, 7)):
            add_row(g1, i, INPUT_NAMES[i], r)

        # 组2：HTT 条件（只放 7-8）
        for r, i in enumerate(range(7, 9)):
            add_row(g2, i, INPUT_NAMES[i], r)

        # ✅ Dilution ratio 单独一组（index=9）
        add_row(g3, 9, INPUT_NAMES[9], 0)

        # 组4：AD 参数（10-11）
        for r, i in enumerate(range(10, 12)):
            add_row(g4, i, INPUT_NAMES[i], r)

        self.out1_var = tk.StringVar()
        self.out2_var = tk.StringVar()

        ttk.Label(output_frame, text=OUTPUT_NAMES[0]).grid(row=0, column=0, sticky="w", pady=6, padx=(0, 10))
        ttk.Entry(output_frame, textvariable=self.out1_var, state="readonly", width=45)\
            .grid(row=0, column=1, sticky="ew", pady=6)

        ttk.Label(output_frame, text=OUTPUT_NAMES[1]).grid(row=1, column=0, sticky="w", pady=6, padx=(0, 10))
        ttk.Entry(output_frame, textvariable=self.out2_var, state="readonly", width=45)\
            .grid(row=1, column=1, sticky="ew", pady=6)

        output_frame.columnconfigure(1, weight=1)

        btn_frame = ttk.Frame(output_frame)
        btn_frame.grid(row=2, column=0, columnspan=2, pady=(12, 0))

        ttk.Button(btn_frame, text="Predict", command=self.predict).grid(row=0, column=0, padx=12)
        ttk.Button(btn_frame, text="Clear", command=self.clear).grid(row=0, column=1, padx=12)

        self.hint = tk.StringVar(value="Please input all features and click Predict")
        ttk.Label(output_frame, textvariable=self.hint, foreground="gray")\
            .grid(row=3, column=0, columnspan=2, sticky="w", pady=(10, 0))

    def _read_inputs(self):
        values = []
        for name, ent in zip(INPUT_NAMES, self.entries):
            txt = ent.get().strip()
            if txt == "":
                raise ValueError(f"{name} is empty.")
            try:
                values.append(float(txt))
            except:
                raise ValueError(f"{name} must be numeric.")
        return np.array(values, dtype=float).reshape(1, -1)

    def predict(self):
        try:
            X = self._read_inputs()
            y = self.model.predict(X)
            if y.shape[1] != 2:
                raise ValueError(f"模型输出维度不是 2 列，而是 {y.shape[1]} 列。请检查训练集 Y 是否为两列。")

            self.out1_var.set(f"{y[0, 0]:.6f}")
            self.out2_var.set(f"{y[0, 1]:.6f}")
            self.hint.set("Prediction completed.")
        except Exception as e:
            messagebox.showerror("Prediction Error", str(e))
            self.hint.set("Prediction failed. Please check inputs.")

    def clear(self):
        for ent in self.entries:
            ent.delete(0, tk.END)
        self.out1_var.set("")
        self.out2_var.set("")
        self.hint.set("Cleared.")


def main():
    # ✅ 这里改成你真实文件名
    excel_file = "Model 2.xlsx"
    try:
        model = train_model_from_excel(excel_file)
    except Exception as e:
        messagebox.showerror("Model Training Error", str(e))
        return

    root = tk.Tk()
    PredictorGUI(root, model)
    root.mainloop()


if __name__ == "__main__":
    main()